# EDA and time-window cleaning

22 November 2026

All 17 CSV logs were reviewed using RPM, MAP and Battery V plots. Event times were checked in MegaLogViewer. Analysis code is in src/eda.py and cleaning code is in src/preprocess.py.

## Keep windows

Settings are stored in data/keep_windows.csv. Times are in seconds. Windows include roughly 3 to 4 seconds around start and shutdown events. Margin by choice for now.

| Log | Start | Shutdown | Keep window |
| --- | ---: | ---: | --- |
| 05092026wires.csv | 281.807 | 665.246 | 278 to 668 |
| 05092026wires2.csv | 14.614 | 685.681 | 11 to 689 |
| 05092026wires3.csv | 69.774 | 931.716 | 66 to 935 |
| 05092026wires4.csv | 32.152 | 824.587 | 29 to 828 |
| final_log.csv | 35.908 | 671.218 | 32 to 675 |
| misfire.csv | 12.070 | 102.486 | 9 to 106 |
| misfire_driving 2.csv | 10.870 | Not recorded | 7 to end |
| misfire_driving latest.csv | 14.335 | 692.064 | 11 to 696 |
| night.csv | Before log | Not recorded | Entire log |
| night2.csv | Before log | 47.238 | Start to 51 |
| random.csv | 69.793 | Not recorded | 66 to end |
| spark_wire_pt2.csv | 38.259 | Key-off at 79.651 | 35 to 83 |
| synclossexperiment.csv | Before log | 794.631 | Start to 798 |
| synclossexperiment2.csv, run 1 | 39.000 | 1399.355 | 36 to 1403 |
| synclossexperiment2.csv, run 2 | 2751.889 | 2869.879 | 2748 to 2873 |
| synclossexperiment3.csv | Before log | 697.413 | Start to 701 |
| synclossexperiment4.csv | 11.175 | Not recorded | 8 to end |
| troubleshoot.csv | Before log | 629.334 | Start to 633 |

## Cleaning results

| Check | Before | After |
| --- | ---: | ---: |
| Logs | 17 | 17 |
| Measurement rows | 255,156 | 150,157 |
| Columns per log | 50 | 50 |
| Missing cells | 2 | 0 |
| Backward time steps | 0 | 0 |

Removed 104,999 rows (41.2%). Outputs are stored in data/processed. Original CSV files remain unchanged. Both runs of synclossexperiment2.csv are saved in one file. Repeated timestamps and short RPM dropouts remain. Later it is decided whether to clean the synclosses away (RPM=0), if they do not serve any purpose.

The two missing Time values (cells) were outside the keep windows and were excluded by the time mask. No values were filled in.

In spark_wire_pt2.csv, 31,500 rows became 722 rows spanning 35.117 to 82.940 s. The plot retains startup, running and shutdown. This is an example log.

The two sections in synclossexperiment2.csv contain 20,630 and 1,887 rows, totalling 22,517.

## Interpretation

- Battery V can read about 12 V with ignition on and the engine stopped. Use RPM and MAP together to recognize running periods.
- Short zero-RPM events during running include sync losses. Keeping them for now. Deciding later whether to remove these rows or use the sync loss channel as a feature.
- The zero-RPM percentage measures rows %, not elapsed time.
- Computer disconnections cause recording gaps. Time continues while sensor values are not recorded. Straight plot lines across gaps indicate that.
- Summary duration is the last timestamp minus the first. It includes gaps and removed middle intervals.
- High AFR and misfire filenames are not descriptive, but random.

## Remaining gaps

| Log | Gap after cleaning (s) | Cause or handling |
| --- | ---: | --- |
| misfire_driving latest.csv | 57.429 | Recording gap |
| night.csv | 567.788 | Recording gap |
| synclossexperiment2.csv | 1345.042 | Removed engine-off interval |
| synclossexperiment4.csv | 66.391 | Recording gap |

Some keep windows end before their upper limit because there are no measurements within the remaining margin:

| Log | Last retained time | Next original time |
| --- | ---: | ---: |
| 05092026wires.csv | 665.246 | 684.701 |
| troubleshoot.csv | 629.334 | 650.227 |

## Next steps

Add and verify segment_id using Time.diff(), dt > 1 and cumsum(). The one-second threshold is provisional. Calculate slopes and rolling features within segments. Do not interpolate across long gaps. Use the source filename and segment ID together when combining logs.

Segment IDs are metadata, not model features. Adding this column gives 51 columns without changing row counts. Implementation has not yet been verified.

Deciding later how to handle repeated timestamps, sync losses and startup/shutdown samples in model training.

## Sources

- Course chapter 3: Reading and plotting.
- Course chapter 4: Preprocessing and feature extraction.
- Pandas: https://pandas.pydata.org/docs/
- DataFrame.plot: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html
- Matplotlib: https://matplotlib.org/stable/

File loops, keep windows, timestamp checks and segmentation are project-specific additions invented.